# JAVDL — Google Colab
Clean test/runner notebook for the `javdl` branch.

**Secrets required:** `TELEGRAM_API_ID`, `TELEGRAM_API_HASH`, `TELEGRAM_BOT_TOKEN`.

Secrets are loaded from Colab Secrets and are never written into the repository.

## 1. Clone the `javdl` branch


## 2. Install system and Python dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg git
!pip install -q -r requirements.txt

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg git
!pip install -q -r requirements.txt

## 1. Clone the `javdl` branch


In [ ]:
import os, shutil
REPO = "https://github.com/Telebotfaroff/javtiful-scraper.git"
BRANCH = "javdl"
WORKDIR = "/content/javdl"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
!git clone -q --depth 1 --branch {BRANCH} {REPO} {WORKDIR}
%cd {WORKDIR}
print("Branch:", BRANCH)


## 3. Load Telegram credentials from Colab Secrets

In Colab: **Secrets → Add new secret** and create the three names below.

In [ ]:
from google.colab import userdata

TELEGRAM_API_ID = userdata.get("TELEGRAM_API_ID")
TELEGRAM_API_HASH = userdata.get("TELEGRAM_API_HASH")
TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")

missing = [name for name, value in {
    "TELEGRAM_API_ID": TELEGRAM_API_ID,
    "TELEGRAM_API_HASH": TELEGRAM_API_HASH,
    "TELEGRAM_BOT_TOKEN": TELEGRAM_BOT_TOKEN,
}.items() if not value]

if missing:
    raise RuntimeError("Missing Colab Secrets: " + ", ".join(missing))

os.environ.update({
    "TELEGRAM_API_ID": str(TELEGRAM_API_ID),
    "TELEGRAM_API_HASH": str(TELEGRAM_API_HASH),
    "TELEGRAM_BOT_TOKEN": str(TELEGRAM_BOT_TOKEN),
    "TELEGRAM_SESSION": "javdl_colab",
    "DEFAULT_UPLOADER": "gofile",
})

print("Telegram credentials loaded from Colab Secrets.")


## 4. Environment check

In [ ]:
import subprocess, shutil

try:
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
except Exception:
    gpu = "No NVIDIA GPU detected"

print("GPU:", gpu or "No NVIDIA GPU detected")
print("FFmpeg:", shutil.which("ffmpeg") or "NOT FOUND")
print("Python:", __import__("sys").version.split()[0])


## 5. Import and syntax test

In [ ]:
!python -m compileall -q app main.py scripts/colab.py
from app.extractor.javtiful import JavtifulExtractor
from app.uploaders.manager import UploadManager

extractor = JavtifulExtractor()
uploads = UploadManager()
print("Extractor:", extractor.name)
print("Upload providers:", uploads.names())


## 6. Test GoFile guest uploader

Set a local test file first. This verifies the GoFile provider without involving Telegram.

In [ ]:
from pathlib import Path
test_file = Path("/content/javdl_gofile_test.txt")
test_file.write_text("JAVDL GoFile guest-upload test")

gofile = uploads.get("gofile")
result = gofile.upload(test_file)
print("Provider:", result["provider"])
print("URL:", result["download_url"])
print("Verified:", gofile.verify(result))


## 7. Test Javtiful extraction

Replace the URL below with a Javtiful post you are authorized to access. This step only extracts metadata and exposed quality URLs.

In [ ]:
JAVTIFUL_URL = ""

if not JAVTIFUL_URL:
    print("Set JAVTIFUL_URL and run this cell to test extraction.")
else:
    video = extractor.extract(JAVTIFUL_URL)
    print("Title:", video.title)
    print("Duration:", video.duration)
    print("Thumbnail:", video.thumbnail)
    print("Qualities:", list(video.qualities))


## 8. Start the Telegram bot

Run this cell only after the environment and provider tests pass. Stop it with the Colab interrupt button.

In [ ]:
from app.bot.telegram import create_app

app = create_app()
print("Starting JAVDL Telegram bot…")
app.run()
